# 02 - Feature Engineering

This notebook demonstrates the calculation of volatility features from 10-minute candles.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from src.data.intraday_features import IntradayFeatureCalculator
from src.data.features import FeatureEngineer

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Calculate Intraday Features

In [ ]:
# Load sample data
DATA_DIR = Path("../../moex_discovery/data/dataset_final")
STOCKS_DIR = DATA_DIR / "01_stocks" / "candles_10m"

TICKER = "SBER"
df = pl.read_parquet(STOCKS_DIR / f"{TICKER}.parquet")
print(f"Loaded {len(df)} bars for {TICKER}")

In [ ]:
# Initialize calculator
calculator = IntradayFeatureCalculator()

# Calculate features
features = calculator.calculate_features_polars(df)
features.head()

## 2. Realized Volatility Components

In [ ]:
features_pd = features.to_pandas()
features_pd['date'] = pd.to_datetime(features_pd['date'])

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# RV
axes[0].plot(features_pd['date'], features_pd['rv'], label='RV', linewidth=0.8)
axes[0].set_ylabel('RV')
axes[0].set_title('Realized Volatility')
axes[0].legend()

# BV
axes[1].plot(features_pd['date'], features_pd['bv'], label='BV', color='orange', linewidth=0.8)
axes[1].set_ylabel('BV')
axes[1].set_title('Bipower Variation')
axes[1].legend()

# Jump
axes[2].bar(features_pd['date'], features_pd['jump'], label='Jump', color='red', alpha=0.7, width=1)
axes[2].set_ylabel('Jump')
axes[2].set_title('Jump Component')
axes[2].legend()

plt.xlabel('Date')
plt.tight_layout()
plt.show()

## 3. Signed Realized Volatility

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(features_pd['date'], features_pd['rsv_pos'], label='RSV+', alpha=0.7)
ax.plot(features_pd['date'], features_pd['rsv_neg'], label='RSV-', alpha=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('RSV')
ax.set_title('Signed Realized Volatility (RSV+ and RSV-)')
ax.legend()

plt.tight_layout()
plt.show()

## 4. Higher Moments

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Realized Skewness
axes[0].plot(features_pd['date'], features_pd['rskew'], linewidth=0.8)
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_ylabel('RSkew')
axes[0].set_title('Realized Skewness')

# Realized Kurtosis
axes[1].plot(features_pd['date'], features_pd['rkurt'], linewidth=0.8, color='green')
axes[1].axhline(3, color='red', linestyle='--', alpha=0.5, label='Normal = 3')
axes[1].set_ylabel('RKurt')
axes[1].set_title('Realized Kurtosis')
axes[1].legend()

plt.xlabel('Date')
plt.tight_layout()
plt.show()

## 5. HAR Features

In [ ]:
# Create HAR lags
features_pd['rv_d'] = features_pd['rv'].shift(1)  # Daily
features_pd['rv_w'] = features_pd['rv'].rolling(5).mean().shift(1)  # Weekly
features_pd['rv_m'] = features_pd['rv'].rolling(22).mean().shift(1)  # Monthly
features_pd['rv_q'] = features_pd['rv'].rolling(66).mean().shift(1)  # Quarterly

# Also for log RV
features_pd['log_rv'] = np.log(features_pd['rv'] + 1e-10)
features_pd['log_rv_d'] = features_pd['log_rv'].shift(1)
features_pd['log_rv_w'] = features_pd['log_rv'].rolling(5).mean().shift(1)
features_pd['log_rv_m'] = features_pd['log_rv'].rolling(22).mean().shift(1)

print("HAR features created:")
features_pd[['date', 'rv', 'rv_d', 'rv_w', 'rv_m', 'rv_q']].dropna().head(10)

In [ ]:
# Correlation between HAR components
har_cols = ['rv', 'rv_d', 'rv_w', 'rv_m', 'rv_q']
corr = features_pd[har_cols].dropna().corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', ax=ax)
ax.set_title('HAR Components Correlation')
plt.tight_layout()
plt.show()

## 6. Feature Statistics

In [ ]:
# Summary statistics for all features
feature_cols = ['rv', 'bv', 'jump', 'rsv_pos', 'rsv_neg', 'rskew', 'rkurt', 'n_bars']
features_pd[feature_cols].describe()

In [ ]:
# Check for missing values
print("Missing values:")
features_pd.isna().sum()

## 7. Create Target Variables

In [ ]:
# Forward targets for different horizons
for h in [1, 5, 22]:
    features_pd[f'y_h{h}'] = features_pd['rv'].shift(-h)
    features_pd[f'log_y_h{h}'] = features_pd['log_rv'].shift(-h)

print("Target variables:")
features_pd[['date', 'rv', 'y_h1', 'y_h5', 'y_h22']].dropna().tail()

## 8. Save Features

In [ ]:
# Save processed features
OUTPUT_DIR = Path("../data/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# features_pd.to_parquet(OUTPUT_DIR / f"{TICKER}_features.parquet", index=False)
# print(f"Saved features to {OUTPUT_DIR / f'{TICKER}_features.parquet'}")

print(f"Features shape: {features_pd.shape}")
print(f"Feature columns: {features_pd.columns.tolist()}")